### Variables, Loops, Functions (with a Sales Example)

When I mentor freshers stepping into the world of data engineering, I notice one common pattern — many try to jump straight into Spark without being comfortable with Python.

That’s like trying to drive a sports car without learning how to steer or brake.

Spark’s core is written in Scala, but for most data engineers, PySpark (Spark’s Python API) is the gateway. And the best part is — you don’t need to be a Python expert. You just need to be comfortable with variables, loops, and functions, because those three ideas control everything you do in Spark.

Today, we’ll revisit them — not in isolation, but in the context of a real Spark job using a tiny dataset.

### Why Python Basics Matter in Spark
Every ETL job you write, at its core, does three things:

- Read the data
- Transform the data
- Write it back to a destination

And inside those steps, Python does the heavy lifting:

- Variables hold dynamic filters, thresholds, and configuration.
- Loops let you repeat logic for multiple inputs.
- Functions capture reusable business logic.

Think of Spark as the engine, and Python as your steering wheel. Without Python basics, you’ll have the power but no control.

Our Sample Dataset: sales.csv
Before diving into code, let’s create a simple CSV you can place in /Volumes/retail/online/ingest/sales.csv (or any folder).



region,store,revenue

North,StoreA,1000

South,StoreB,1500

East,StoreC,1200

West,StoreD,

North,StoreE,900

South,StoreF,NA

East,StoreG,1100

West,StoreH,800

Notice there are missing revenues — this will help us practice data cleaning later.

###  Variables in Action

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("day2-demo").getOrCreate()
df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/sales/sales1.csv")

target_region = "North"

df_filtered = df.filter(df["region"] == target_region)
df_filtered.show()

Here, target_region controls which part of the dataset Spark processes. Change it to "East" or "South" — the logic remains the same.

### Loops to Automate Logic

In [0]:
regions = ["North", "South", "East", "West"]

for r in regions:
    count = df.filter(df["region"] == r).count()
    print(f"{r} region has {count} records.")

Instead of four separate queries, one simple loop does it all.
 This is the pattern you’ll use for batch jobs across multiple files or tables.

### Functions to Reuse Logic

Functions organize logic into small reusable blocks. Let’s define one to clean and enrich our data.

In [0]:
from pyspark.sql.functions import col, avg, trim

df_dirty = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/thedataengineering_00/data/sampledata/salesdirty/salesdirty.csv")
df_dirty.show()

def fill_missing_revenue(df):

    # Clean the values from Dataframe
    df_clean = df.filter(
    (trim(col("revenue")) != "") &              # drop empty strings or spaces
    (trim(col("revenue")) != "NA")              # drop literal "NA" 
    )
                       
    # Make sure revenue is numeric
    df_clean = df_clean.withColumn("revenue", col("revenue").cast("double"))
    # Compute average
    avg_rev = df_clean.select(avg(col("revenue"))).first()[0]
    print(f"Average revenue used for imputation: {avg_rev}")


    # Fill missing values
    df_imputed = df_clean.na.fill({"revenue": avg_rev})
    return df_imputed

In [0]:
df_imputed = fill_missing_revenue(df_dirty)
df_imputed.show()

This small function did three things for us:

- Converted revenue into a numeric type.
- Computed the average.
- Filled all missing values with that average.

You’ve just written your first data-cleaning transformation in Spark — entirely controlled by Python.

This pattern — loop + function + variable — is exactly how production ETL jobs are structured.
 You might read multiple source files, apply a cleaning function to each, and then write the result to your Delta tables.

## Wrapping Up
You’ve just used Python’s three essentials inside a Spark workflow:

- Variables control what part of data Spark processes.
- Loops automate repetitive operations.
- Functions package business logic into reusable building blocks.

The next time you open a PySpark notebook, remember this: Spark is powerful because of Python — not in spite of it.
 
Master these basics, and you’ll have full control over how your Spark jobs behave.